In [1]:
import numpy as np
import pandas as pd
import joblib

In [13]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Window
from pyspark import SparkConf

In [3]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/07 18:13:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
ftr_transactions = spark.read.parquet("../data/04_feature/ftr_transactions")

In [5]:
ftr_transactions.columns

['_id',
 '_observ_end_dt',
 'month_total_sales',
 'month_net_sales',
 'month_total_profit',
 'month_total_cost',
 'month_total_amount_discounts',
 'month_total_qty_discounts',
 'month_distinct_skus_sold',
 'month_distinct_branches',
 'month_distinct_product_categories',
 'month_distinct_transactions',
 'month_avg_order',
 'month_avg_skus_per_order',
 'month_net_sales_category_ac',
 'month_total_profit_category_ac',
 'month_total_cost_category_ac',
 'month_total_amount_discounts_category_ac',
 'month_total_qty_discounts_category_ac',
 'month_net_sales_category_additives',
 'month_total_profit_category_additives',
 'month_total_cost_category_additives',
 'month_total_amount_discounts_category_additives',
 'month_total_qty_discounts_category_additives',
 'month_net_sales_category_air_filter',
 'month_total_profit_category_air_filter',
 'month_total_cost_category_air_filter',
 'month_total_amount_discounts_category_air_filter',
 'month_total_qty_discounts_category_air_filter',
 'month_net_

In [18]:
w_churn_acc = Window.partitionBy("_id").orderBy(f.col("_observ_end_dt")).rowsBetween(-2, 0)

win_id_monthly = Window.partitionBy("_id").orderBy(f.col("_observ_end_dt"))

In [25]:
ftr_transactions = ftr_transactions.withColumn(
    "days_until_mineral_oil_change",
    f.col("estimated_days_to_change_mineral_oil") - f.col("days_since_last_transactions")
).withColumn(
    "days_until_synthetic_oil_change",
    f.col("estimated_days_to_change_synthetic_oil") - f.col("days_since_last_transactions")
).withColumn(
    "below_estimated_mineral_oil_change",
    (f.col("days_until_mineral_oil_change") > 0).cast("int")
).withColumn(
    "below_estimated_synthetic_oil_change",
    (f.col("days_until_synthetic_oil_change") > 0).cast("int")
).withColumn(
    "customer_mineral_oil",
    f.when(
        f.col("month_net_sales_category_oil") > 0,
        1
    ).otherwise(None)
).withColumn(
    "customer_synthetic_oil",
    f.when(
        f.col("month_net_sales_category_oil_synthetic") > 0,
        1
    ).otherwise(None)
).withColumn(
    "customer_mineral_oil",
    f.last("customer_mineral_oil", ignorenulls=True).over(win_id_monthly.rangeBetween(Window.unboundedPreceding, 0))
).withColumn(
    "customer_synthetic_oil",
    f.last("customer_synthetic_oil", ignorenulls=True).over(win_id_monthly.rangeBetween(Window.unboundedPreceding, 0))
).withColumn(
    "customer_mineral_oil",
    f.when(
        f.col("customer_mineral_oil").isNull(),
        0
    ).otherwise(f.col("customer_mineral_oil"))
).withColumn(
    "customer_synthetic_oil",
    f.when(
        f.col("customer_synthetic_oil").isNull(),
        0
    ).otherwise(f.col("customer_synthetic_oil"))
).withColumn(
    "is_churn",
    f.when(
        (f.col("is_active") > 0),
        0
    ).when(
        (f.col("customer_mineral_oil") > 0) &
        (f.col("below_estimated_mineral_oil_change") > 0),
        0
    ).when(
        (f.col("customer_synthetic_oil") > 0) &
        (f.col("below_estimated_synthetic_oil_change") > 0),
        0
    ).otherwise(1)
).withColumn(
    "is_churn_30",
    f.when(
        (f.col("is_active") > 0),
        0
    ).when(
        (f.col("customer_mineral_oil") > 0) &
        (f.col("days_until_mineral_oil_change") < 30),
        0
    ).when(
        (f.col("customer_synthetic_oil") > 0) &
        (f.col("days_until_synthetic_oil_change") < 30),
        0
    ).otherwise(1)
).withColumn(
    "is_churn_60",
    f.when(
        (f.col("is_active") > 0),
        0
    ).when(
        (f.col("customer_mineral_oil") > 0) &
        (f.col("days_until_mineral_oil_change") < 60),
        0
    ).when(
        (f.col("customer_synthetic_oil") > 0) &
        (f.col("days_until_synthetic_oil_change") < 60),
        0
    ).otherwise(1)
).withColumn(
    "target_churn_1",
    f.max("is_churn").over(win_id_monthly.rowsBetween(1, 1))
).withColumn(
    "target_churn_2",
    f.max("is_churn").over(win_id_monthly.rowsBetween(2, 2))
).withColumn(
    "target_churn_3",
    f.max("is_churn").over(win_id_monthly.rowsBetween(3, 3))
).withColumn(
    "acc_churn_past_2_next_0_months",
    f.sum("is_churn").over(w_churn_acc)
)

In [26]:
ftr_transactions.filter(
    f.col("is_active") > 0
).filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2024-01-01"
).groupBy(
    "_observ_end_dt"
).agg(
    f.countDistinct("_id").alias("# of clients"),
    f.sum("customer_mineral_oil").alias("# of mineral oil clients"),
    f.sum("customer_synthetic_oil").alias("# of synthetic oil clients"),
).orderBy(
    "_observ_end_dt"
).show()

+--------------+------------+------------------------+--------------------------+
|_observ_end_dt|# of clients|# of mineral oil clients|# of synthetic oil clients|
+--------------+------------+------------------------+--------------------------+
|    2021-01-31|      258787|                  173476|                     84766|
|    2021-02-28|      223925|                  150981|                     73196|
|    2021-03-31|      262208|                  176877|                     86609|
|    2021-04-30|      235781|                  159294|                     78148|
|    2021-05-31|      289906|                  195006|                     99062|
|    2021-06-30|      265862|                  173911|                     95122|
|    2021-07-31|      286124|                  189636|                    100888|
|    2021-08-31|      288960|                  192054|                    101731|
|    2021-09-30|      341952|                  241378|                    112854|
|    2021-10-31|

In [27]:
ftr_transactions.filter(
    # f.col("is_active") > 0
    f.lit(True) == True
).filter(
    f.col("acc_churn_past_2_next_0_months") < 2
    # f.lit(True) == True
).filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2024-01-01"
).groupBy(
    "_observ_end_dt"
).agg(
    f.mean("target_churn_1").alias("target_1_pos"),
    f.mean("target_churn_2").alias("target_2_pos"),
    f.mean("target_churn_3").alias("target_3_pos"),
).orderBy("_observ_end_dt").show(50, truncate=False)

25/05/07 18:35:30 WARN Column: Constructing trivially true equals predicate, 'true = true'. Perhaps you need to use aliases.


+--------------+-------------------+-------------------+-------------------+
|_observ_end_dt|target_1_pos       |target_2_pos       |target_3_pos       |
+--------------+-------------------+-------------------+-------------------+
|2021-01-31    |0.48639221833899526|0.5446970675623305 |0.5981191758869061 |
|2021-02-28    |0.46010763235133845|0.5364499362999876 |0.5513452164860894 |
|2021-03-31    |0.45861352667235233|0.5014140418494781 |0.5367025198630044 |
|2021-04-30    |0.43261535027582027|0.49135271515576673|0.5213740792704314 |
|2021-05-31    |0.42534586372030425|0.479764624870888  |0.5134296065604558 |
|2021-06-30    |0.42016137276800786|0.47717422125005066|0.4970979520742554 |
|2021-07-31    |0.41567792294023775|0.45974791279186017|0.5152640653117891 |
|2021-08-31    |0.4000387053794895 |0.4781063393084763 |0.5196699622064299 |
|2021-09-30    |0.41189752209056607|0.5030128254304783 |0.5486567819955784 |
|2021-10-31    |0.43690869895613943|0.5035754837132599 |0.5433176450165533 |

In [7]:
ftr_transactions.filter(
    f.col("is_active") > 0
).filter(
    f.col("month_avg_mileage_per_day") < 101
    # f.lit(True) == True
).filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2024-01-01"
).show(5, truncate=False)

25/05/07 18:14:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------------------------------------------------------------+--------------+------------------+------------------+------------------+------------------+----------------------------+-------------------------+------------------------+-----------------------+---------------------------------+---------------------------+------------------+------------------------+---------------------------+------------------------------+----------------------------+----------------------------------------+-------------------------------------+----------------------------------+-------------------------------------+-----------------------------------+-----------------------------------------------+--------------------------------------------+-----------------------------------+--------------------------------------+------------------------------------+------------------------------------------------+---------------------------------------------+----------------------------------+------------

In [8]:
working_ids = [
    "A4F50776-4238-465E-AE0D-1C8C376CAEE2__37D422A0-E176-4D4F-8721-3B231E1ABAA0",
    "A4F507F5-769F-4C5E-9D3A-CBDD668AB6C8__4124CB72-D524-4A5E-8806-2D741083C98A",
    "A4F508D3-F915-4881-92E1-64E425B7FA71__DE8981C4-D7CF-46AC-8E18-5A844E1B5F7F",
    "A4F510D7-F015-432A-ABCB-E72594B42014__BD2B7690-017A-4437-9F8C-85518A3CF131",
    "A4F51822-9DBE-4CEF-834F-F5BC2DC179EC__B089C48B-85D1-4E8C-BBFC-33BEA68D5027",
    "A4F52279-0F64-4165-BB89-600CD3314D59__4D5A9841-55D7-492F-B695-78DE31B6A760", ## GOOD ID
    "A4F553E8-D26B-4BB7-90E3-75D3F2950064__D71989F6-16FF-48D5-919B-4BA2286EF08D", ## GOOD ID
]

In [21]:
ftr_transactions.filter(
    # f.col("is_active") > 0
    f.col("_id") == working_ids[6]
).filter(
    # f.col("acc_churn_past_2_next_0_months") < 2
    f.lit(True) == True
).filter(
    f.col("_observ_end_dt") >= "2021-01-01"
).filter(
    f.col("_observ_end_dt") < "2024-01-01"
).select("_id", "_observ_end_dt", "is_active", "customer_mineral_oil", "customer_synthetic_oil", "last_trx_total_sales", "month_total_sales", "last_trx_current_mileage", "month_avg_mileage_per_day", "month_avg_mileage_per_day_max_past_5_next_0_months", "month_avg_mileage_per_day_nullif_past_11_next_0_months", "below_estimated_mineral_oil_change", "below_estimated_synthetic_oil_change", "is_churn", "target_churn_1", "acc_churn_past_2_next_0_months").orderBy("_id", "_observ_end_dt").show(30, truncate=False)

25/05/07 18:26:05 WARN Column: Constructing trivially true equals predicate, 'true = true'. Perhaps you need to use aliases.


+--------------------------------------------------------------------------+--------------+---------+--------------------+----------------------+--------------------+-----------------+------------------------+-------------------------+--------------------------------------------------+------------------------------------------------------+----------------------------------+------------------------------------+--------+--------------+------------------------------+
|_id                                                                       |_observ_end_dt|is_active|customer_mineral_oil|customer_synthetic_oil|last_trx_total_sales|month_total_sales|last_trx_current_mileage|month_avg_mileage_per_day|month_avg_mileage_per_day_max_past_5_next_0_months|month_avg_mileage_per_day_nullif_past_11_next_0_months|below_estimated_mineral_oil_change|below_estimated_synthetic_oil_change|is_churn|target_churn_1|acc_churn_past_2_next_0_months|
+-------------------------------------------------------------

In [6]:
mdl_churn_avg_metrics = joblib.load("../data/06_models/churn/model/mdl_churn_avg_metrics.pkl")
mdl_churn_metrics_df = pd.read_parquet("../data/06_models/churn/model/mdl_churn_metrics_df")
mdl_selected_features = joblib.load("../data/06_models/churn/model/selected_features.pkl")

In [4]:
mdl_churn_avg_metrics

{'precision': 0.6481185410646705,
 'recall': 0.6419665879330496,
 'roc_auc': 0.7761207820030527,
 'pr_auc': 0.7538142690267893,
 'ks': 0.4107926440628127,
 'tp': 215906.7,
 'tn': 222386.6,
 'fp': 112524.7,
 'fn': 86541.0}

In [5]:
mdl_churn_metrics_df

,iteration,train_period,test_period,precision,recall,roc_auc,pr_auc,ks,ks-pvalue,tp,tn,fp,fn
0,1,2021-01-01/2022-01-01,2022-07-01/2022-07-01,0.726571,0.709551,0.771274,0.790357,0.404637,0.0,252881,215960,95166,103515
1,2,2021-01-01/2022-04-01,2022-10-01/2022-10-01,0.691943,0.704755,0.761344,0.751346,0.389048,0.0,223279,215236,99405,93539
2,3,2021-01-01/2022-07-01,2023-01-01/2023-01-01,0.689537,0.737085,0.764680,0.757771,0.395610,0.0,239361,205288,107772,85379
3,4,2021-01-01/2022-10-01,2023-04-01/2023-04-01,0.699407,0.715777,0.770952,0.765116,0.402968,0.0,227927,214260,97959,90506
4,5,2021-01-01/2023-01-01,2023-07-01/2023-07-01,0.738966,0.724550,0.786610,0.806055,0.426319,0.0,259686,215264,91732,98724
5,6,2021-01-01/2023-04-01,2023-10-01/2023-10-01,0.718889,0.678084,0.772169,0.769887,0.403343,0.0,213083,218571,83323,101160
6,7,2021-01-01/2023-07-01,2024-01-01/2024-01-01,0.721685,0.704370,0.776616,0.775790,0.411890,0.0,226791,211350,87461,95186
7,8,2021-01-01/2023-10-01,2024-04-01/2024-04-01,0.722038,0.714242,0.778859,0.787053,0.412910,0.0,237674,211926,91497,95090
8,9,2021-01-01/2024-01-01,2024-07-01/2024-07-01,0.772148,0.731253,0.802583,0.834768,0.450409,0.0,278385,209722,82148,102311
9,10,2021-01-01/2024-04-01,2024-10-01/2024-10-01,0.000000,0.000000,NaN,0.500000,NaN,NaN,0,306289,288784,0


In [8]:
mdl_churn_metrics_df.iloc[:-1].drop(columns=["train_period", "test_period"]).mean()

iteration         5.000000
precision         0.720132
recall            0.713296
roc_auc           0.776121
pr_auc            0.782016
ks                0.410793
ks-pvalue         0.000000
tp           239896.333333
tn           213064.111111
fp            92940.333333
fn            96156.666667
dtype: float64

In [1]:
(239896.333333 + 96156.666667) / (239896.333333 + 213064.111111 + 92940.333333 + 96156.666667)

0.5234002080468213

In [9]:
239896.333333 + 213064.111111 + 92940.333333 + 96156.666667

642057.444444

In [10]:
239896 / 642057.444444, 213064 / 642057.444444, 92940 / 642057.444444, 96157 / 642057.444444

(0.37363634994956224,
 0.33184569674214465,
 0.1447534029925981,
 0.14976385809725903)

In [37]:
mdl_churn_predicted_df = pd.read_parquet("../data/07_model_output/churn/predict/mdl_churn_predicted")


In [38]:
mdl_churn_predicted_df_filtered = mdl_churn_predicted_df[["_id","is_active", "churn_probability"]]
# mdl_churn_predicted_df_filtered[["customer_id", "customer_vehicle_id"]] = mdl_churn_predicted_df_filtered["_id"].str.split("__")
mdl_churn_predicted_df_filtered[["customer_id", "customer_vehicle_id"]] = mdl_churn_predicted_df_filtered["_id"].str.rpartition("__")[[0,2]]

/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_40882/4259710272.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mdl_churn_predicted_df_filtered[["customer_id", "customer_vehicle_id"]] = mdl_churn_predicted_df_filtered["_id"].str.rpartition("__")[[0,2]]
/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_40882/4259710272.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mdl_churn_predicted_df_filtered[["customer_id", "customer_vehicle_id"]] = mdl_churn_predicted_df_filtered["_id"].st

In [39]:
mdl_churn_predicted_df_filtered.head()

,_id,is_active,churn_probability,customer_id,customer_vehicle_id
0,00000540-7986-4B85-A946-E1A9D913344F__F40BF538...,0,0.699744,00000540-7986-4B85-A946-E1A9D913344F,F40BF538-A9A6-49E3-9623-9301AE9FD953
1,00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321...,0,0.708705,00000847-CECD-43FB-A2F9-4D4B44A8BFFC,FF645321-C6D1-4864-91B7-F00EA37C30DB
2,00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477...,0,0.679837,00000E2B-FD56-4F34-A54C-159E564F5F79,FF8A2477-4270-4547-B4CB-5706EA4479A0
3,00001CA5-294A-4B26-A4A1-A77D7B5B18CC__9946561B...,1,0.637760,00001CA5-294A-4B26-A4A1-A77D7B5B18CC,9946561B-097B-441A-BB89-95AF12D27759
4,00001DF1-C75B-4CFA-9D95-FB370E5B95A7__E83698A7...,0,0.593648,00001DF1-C75B-4CFA-9D95-FB370E5B95A7,E83698A7-E358-42E9-9052-57664B17C455


In [40]:
mdl_churn_predicted_df_filtered.shape

(3402552, 5)

In [41]:
# mdl_churn_predicted_df_filtered.to_excel("../../../predict_20250401.xlsx", engine="openpyxl")

In [42]:
customer_df = pd.read_parquet("../data/01_raw/customers.parquet")
vehicle_df = pd.read_parquet("../data/01_raw/vehicles.parquet")

In [43]:
customer_df.head()

,CustomerID,CreationOn,ModifiedOn,UserTitle,CustomerName,Mobile,Mobile2,Email,Gender,Nationality,BirthDate,LocationName,IsOwner,IsCashCustomer,PayTerm,Age,IsActive,StationBrand
0,EB5670AA-72D8-49CE-8FAC-00000859EB85,2021-03-12 00:11:18,2020-03-26 11:55:24,None,ABU RAKAN,0555551397,None,None,None,None,NaN,None,True,True,None,NaN,True,PE
1,4C86E939-005E-4611-B4C1-00000A03F8CA,2021-03-12 00:11:18,2020-03-26 11:55:24,None,MR OLAYYAN,0547502121,None,None,None,None,NaN,None,True,True,None,NaN,True,PE
2,F6090720-12AA-4DC3-BA2A-00001D49C2DC,2021-03-12 00:11:18,2020-03-26 11:55:24,None,MR.AWAD,0534077960,None,None,None,None,NaN,None,True,True,None,NaN,True,PE
3,5E65098C-E166-466D-9309-000022944B45,2021-03-12 00:11:18,2020-03-26 11:55:24,None,MR. MOHAMAD MOBARAK DOSERI,0546919905,None,None,None,None,NaN,None,True,True,None,NaN,True,PE
4,E6E57568-A4F0-4A0D-95DD-00002CA50962,2021-03-12 00:11:18,2020-03-26 11:55:24,None,MR.ALI,0503814721,None,None,None,None,NaN,None,True,True,None,NaN,True,PE


In [44]:
vehicle_df.head()

,CustomerVehicleID,CreationOn,ModifiedOn,CustomerID,Make,Model,ModelYear,TransmissionType,PMSName,PlateNumber,VIN,IsDriverOwner,StationBrand,is_truck,vehicle_brand_level
0,918FE112-E062-4A56-8033-000001BFCEC6,2021-03-12 00:11:44,2020-03-26 11:55:24,40AA5E88-1966-4494-B3FE-54112BF3F479,chevrolet,SUBURBAN,2013.0,Automatic,None,5340AAJ,1GNSC8E07DR219313,False,PE,0,low
1,1B4AA1DD-1E18-4990-AB45-000002BF3307,2021-03-12 00:11:44,2020-03-26 11:55:24,E1EAFA23-0ADC-4AFA-A774-4E27CDA23AA9,lexus,LS400,1998.0,Automatic,TOYOTA,0068LAZ,JTB530F20W0115484,False,PE,0,high
2,8752306B-7FD4-4ABB-8E52-00002A049E37,2021-03-12 00:11:44,2020-03-26 11:55:24,22B51812-64A4-44F1-A5AD-F6850639C800,toyota,HILLUX,2016.0,Automatic,TOYOTA,8412SBB,MR0HX8CD3G0902343,False,PE,0,high
3,CECB08B1-73B0-44A7-B19F-00002BE80CC8,2021-03-12 00:11:44,2020-03-26 11:55:24,C54F5632-2621-4BD5-8040-6CF1A92BA3DA,hyundai,ACCENT,2009.0,Automatic,HYUNDAI,9261RKA,KMHCM41A89U326551,False,PE,0,other
4,286358CB-BE12-40D8-A428-0000425C16D7,2021-03-12 00:11:44,2020-03-26 11:55:24,7B7E7432-17D1-4F57-BE02-D0CD61CEE722,hyundai,Elantra,2017.0,Automatic,HYUNDAI,4470UAD,KMHD741F4HU156825,False,PE,0,other


In [45]:
mdl_churn_predicted_df_extended = pd.merge(
    mdl_churn_predicted_df_filtered,
    customer_df[["CustomerID", "CustomerName", "Mobile"]],
    left_on="customer_id",
    right_on="CustomerID"
).drop(columns="CustomerID")

# mdl_churn_predicted_df_extended.head()

In [46]:
mdl_churn_predicted_df_extended = pd.merge(
    mdl_churn_predicted_df_extended,
    vehicle_df[["CustomerID", "CustomerVehicleID", "PlateNumber", "VIN"]],
    left_on=["customer_id", "customer_vehicle_id"],
    right_on=["CustomerID", "CustomerVehicleID"]
).drop(columns=["CustomerID", "CustomerVehicleID"])

mdl_churn_predicted_df_extended.head()

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN
0,00000540-7986-4B85-A946-E1A9D913344F__F40BF538...,0,0.699744,00000540-7986-4B85-A946-E1A9D913344F,F40BF538-A9A6-49E3-9623-9301AE9FD953,TARIQ ABDUL AZIZ,0534747726,1065KHD,TMAJ28130MJ152970
1,00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321...,0,0.708705,00000847-CECD-43FB-A2F9-4D4B44A8BFFC,FF645321-C6D1-4864-91B7-F00EA37C30DB,MR. ALI MOHAMMAD ASIRI,0507340298,9664EHJ,6T1BF9FK7GX648439
2,00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477...,0,0.679837,00000E2B-FD56-4F34-A54C-159E564F5F79,FF8A2477-4270-4547-B4CB-5706EA4479A0,MOHAMAD,0548121618,2250ASR,KNAF241BEKA835999
3,00001CA5-294A-4B26-A4A1-A77D7B5B18CC__9946561B...,1,0.637760,00001CA5-294A-4B26-A4A1-A77D7B5B18CC,9946561B-097B-441A-BB89-95AF12D27759,MR.FAIF,0571317927,8123UEB,KMHDT41B5BU252737
4,00001DF1-C75B-4CFA-9D95-FB370E5B95A7__E83698A7...,0,0.593648,00001DF1-C75B-4CFA-9D95-FB370E5B95A7,E83698A7-E358-42E9-9052-57664B17C455,HAMAM ALI,0557546128,3902NSB,VR3EF9HPANJ529993


In [ ]:
customer_ago_df = pd.read_excel("../../../May 2025 Personalized Campaign Audiences.xlsx", )

In [48]:
customer_ago_df.head()

,Mobile,PlateNumber,Make,Model,Nationality,PreferredLanguage,Campaign,Segment,IsPromoHunter,LastPromoTaken
0,966559731819,1349KSD,HYUNDAI,SANTA FE,Saudi,AR,PAC Brakes,Potential Loyal,0,NaN
1,966562041000,7842LXB,KIA,CERATO,Egyptian,AR,PAC Brakes,Potential Loyal,1,Saudi Foundation Day 2025 Mineral Oils
2,966532013322,3843ASR,TOYOTA,RUSH,Others,AR,PAC Brakes,Loyal,0,NaN
3,966532932686,4031SGR,HYUNDAI,SANTA FE,Saudi,AR,PAC Brakes,Potential Loyal,0,NaN
4,966504539322,7063DJR,MAZDA,MAZDA 6,Saudi,AR,PAC Brakes,Loyal,0,NaN


In [49]:
mdl_churn_predicted_df_extended.shape

(3402552, 9)

In [50]:
customer_ago_df["Mobile"] = customer_ago_df["Mobile"].astype("string")

In [59]:
mdl_churn_predicted_df_extended["PlateNumber"] = mdl_churn_predicted_df_extended["PlateNumber"].str.zfill(7)

In [60]:
mdl_churn_predicted_df_202505 = pd.merge(
    mdl_churn_predicted_df_extended,
    customer_ago_df, #[["Mobile", "PlateNumber"]],
    left_on=["PlateNumber"],
    right_on=["PlateNumber"]
)

mdl_churn_predicted_df_202505.head()

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile_x,PlateNumber,VIN,Mobile_y,Make,Model,Nationality,PreferredLanguage,Campaign,Segment,IsPromoHunter,LastPromoTaken
0,0005650E-C752-4B05-A1C0-2719F69264FE__65C91464...,1,0.386644,0005650E-C752-4B05-A1C0-2719F69264FE,65C91464-BE6C-47E7-9F8F-05B280A9E0D5,MR. ABDUIRAHMAN,0506454227,5735HGD,KMHL341BXLA042682,966506454227,HYUNDAI,SONATA,Saudi,AR,PAC Brakes,Loyal,0,NaN
1,00056963-1D78-48AC-9B4B-66264186AFB1__11D4B8D2...,0,0.544847,00056963-1D78-48AC-9B4B-66264186AFB1,11D4B8D2-97A0-4AF4-83E1-F5A4102406CA,MR.ANAS,0536793734,2779JXD,RKLBB9HE7J5200825,966536793734,TOYOTA,COROLLA,Others,AR,PAC Brakes,Potential Loyal,0,NaN
2,0006B1F0-93AB-4DE0-8D83-8B1F7FE4354D__A5172E55...,1,0.551570,0006B1F0-93AB-4DE0-8D83-8B1F7FE4354D,A5172E55-3179-44B4-AFE6-5C37BBFF119D,SULAIMAN ABDULAZIZ,0500395119,7544NTB,6T1BE42K4BX706897,966500395119,TOYOTA,CAMRY,Saudi,AR,PAC Brakes,Outliers,0,NaN
3,00144005-7562-43A7-83B7-E790BB3A7BB5__19C11E01...,0,0.330104,00144005-7562-43A7-83B7-E790BB3A7BB5,19C11E01-D212-4212-90BF-E1D7E5AA8EE1,ABDULLILAH,0508820232,5172GRR,JM5TCAWY2P0472152,966508820232,MAZDA,MAZDA CX-9,Saudi,AR,PAC Brakes,Loyal,0,NaN
4,0016DA26-209F-4552-A129-C78048AD7B9F__ADEC36A7...,1,0.760857,0016DA26-209F-4552-A129-C78048AD7B9F,ADEC36A7-C019-4ACE-926E-B3892EC5238C,MR. MUHAMMAD,0591313611,7522DKD,JM7KFAWL4L0361045,966591313611,MAZDA,CX-5,Pakistani,EN,PAC Brakes,Potential Loyal,0,NaN


In [61]:
mdl_churn_predicted_df_202505.shape

(13390, 18)

In [53]:
filter_mobile = mdl_churn_predicted_df_extended["PlateNumber"].isin(customer_ago_df["PlateNumber"].unique())
mdl_churn_predicted_df_extended[filter_mobile]

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN
263,0005650E-C752-4B05-A1C0-2719F69264FE__65C91464...,1,0.386644,0005650E-C752-4B05-A1C0-2719F69264FE,65C91464-BE6C-47E7-9F8F-05B280A9E0D5,MR. ABDUIRAHMAN,0506454227,5735HGD,KMHL341BXLA042682
267,00056963-1D78-48AC-9B4B-66264186AFB1__11D4B8D2...,0,0.544847,00056963-1D78-48AC-9B4B-66264186AFB1,11D4B8D2-97A0-4AF4-83E1-F5A4102406CA,MR.ANAS,0536793734,2779JXD,RKLBB9HE7J5200825
339,0006B1F0-93AB-4DE0-8D83-8B1F7FE4354D__A5172E55...,1,0.551570,0006B1F0-93AB-4DE0-8D83-8B1F7FE4354D,A5172E55-3179-44B4-AFE6-5C37BBFF119D,SULAIMAN ABDULAZIZ,0500395119,7544NTB,6T1BE42K4BX706897
1041,00144005-7562-43A7-83B7-E790BB3A7BB5__19C11E01...,0,0.330104,00144005-7562-43A7-83B7-E790BB3A7BB5,19C11E01-D212-4212-90BF-E1D7E5AA8EE1,ABDULLILAH,0508820232,5172GRR,JM5TCAWY2P0472152
1171,0016DA26-209F-4552-A129-C78048AD7B9F__ADEC36A7...,1,0.760857,0016DA26-209F-4552-A129-C78048AD7B9F,ADEC36A7-C019-4ACE-926E-B3892EC5238C,MR. MUHAMMAD,0591313611,7522DKD,JM7KFAWL4L0361045
...,...,...,...,...,...,...,...,...,...
3401104,FFE49831-397B-459E-98F0-B9FF89610599__760C18DB...,1,0.782909,FFE49831-397B-459E-98F0-B9FF89610599,760C18DB-9A28-4C9F-B1E3-01B23C962F19,ABDUL RAHMAN,0555856863,6248ZDB,1FTMF1C58JFC74789
3401160,FFE5B988-F3FA-4BE8-A3EF-397149AC7B4F__41F4B34D...,1,0.657453,FFE5B988-F3FA-4BE8-A3EF-397149AC7B4F,41F4B34D-101A-4925-BBAC-4BBCB462072E,MR. HAMMAD TARIK,0534910424,9121NKD,MDHBN7AD7LG709296
3401613,FFEDF978-FEE0-4BCB-9679-81F87C7A8E24__BB4CBEAD...,1,0.693291,FFEDF978-FEE0-4BCB-9679-81F87C7A8E24,BB4CBEAD-6E1C-40C5-B33F-5D12BC703B26,MOHAMMAD WAJID HUSSAIN,0566049905,6206VKR,LSJW74U38PZ050277
3401718,FFF00F54-7239-4AA0-950E-F9038BEC5850__877C923A...,1,0.507788,FFF00F54-7239-4AA0-950E-F9038BEC5850,877C923A-5694-4F7C-99C0-0781A7FDA052,MR ABDULLAH,0530555554,9581DZR,JTDZ43FV5RD194580


In [75]:
mdl_churn_predicted_df_202505.to_excel("../../../predict_ago_table_20250401.xlsx", engine="openpyxl")

In [62]:
filter_mobile = ~customer_ago_df["PlateNumber"].isin(mdl_churn_predicted_df_extended["PlateNumber"].unique())
customer_ago_df[filter_mobile].head()

,Mobile,PlateNumber,Make,Model,Nationality,PreferredLanguage,Campaign,Segment,IsPromoHunter,LastPromoTaken
185,966558109090,5675VGD,HYUNDAI,ACCENT,Saudi,AR,PAC Brakes,Outliers,0,NaN
463,966564077140,8464RTD,NISSAN,SUNNY (N17),Others,EN,PAC Brakes,Outliers,1,Saudi Foundation Day 2025 Mineral Oils
472,966558498251,9617XAR,HYUNDAI,ACCENT,Others,EN,PAC Brakes,Outliers,0,NaN
729,966504325339,8065SUD,CHANGAN,EADO PLUS,Pakistani,EN,PAC Brakes,Outliers,0,NaN
1149,966554655130,41111AD,JEEP,Gladiator,Omani,AR,PAC Brakes,Loyal,0,NaN


In [72]:
filter_mobile = mdl_churn_predicted_df_extended["PlateNumber"].str.contains("8464RTD")
mdl_churn_predicted_df_extended[filter_mobile]

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN


In [73]:
filter_mobile = mdl_churn_predicted_df_extended["Mobile"].str.contains("4077140")
mdl_churn_predicted_df_extended[filter_mobile]

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN


In [36]:
mdl_churn_predicted_df_extended.to_excel("../../../predict_extended_20250401.xlsx", engine="openpyxl")

In [37]:
branch_df = pd.read_parquet("../data/01_raw/branches.parquet")

In [39]:
branch_df.isna().sum()

BranchID             0
BranchCode           0
BranchType          77
IsActive             0
BranchName           0
AMBadgeNo           98
AreaManager         98
Region              21
RMBadgeNo          155
RegionalManager    155
Longitude          136
Latitude           136
City                24
StationBrand         0
dtype: int64

In [40]:
branch_df.to_excel("../../../branches_information_20250427.xlsx", engine="openpyxl")

In [89]:
resty_PAC_df = pd.read_excel("Resty Original File.xlsx", sheet_name="PAC")
resty_PE_df = pd.read_excel("Resty Original File.xlsx", sheet_name="PE")
saima_df = pd.read_excel("Saima Revisited File.xlsx")
resty_PAC_df["file"] = "PAC"
resty_PE_df["file"] = "PE"
resty_df = pd.concat([resty_PAC_df, resty_PE_df], axis=0)

In [90]:
resty_df.head()

,Mobile,PlateNumber,Make,Model,Nationality,PreferredLanguage,Campaign,Segment,IsPromoHunter,LastPromoTaken,file,IsChinesePMS,DueOil,DuePMS,DueAirFilter
0,966559731819,1349KSD,HYUNDAI,SANTA FE,Saudi,AR,PAC Brakes,Potential Loyal,0,NaN,PAC,NaN,NaN,NaN,NaN
1,966562041000,7842LXB,KIA,CERATO,Egyptian,AR,PAC Brakes,Potential Loyal,1,Saudi Foundation Day 2025 Mineral Oils,PAC,NaN,NaN,NaN,NaN
2,966532013322,3843ASR,TOYOTA,RUSH,Others,AR,PAC Brakes,Loyal,0,NaN,PAC,NaN,NaN,NaN,NaN
3,966532932686,4031SGR,HYUNDAI,SANTA FE,Saudi,AR,PAC Brakes,Potential Loyal,0,NaN,PAC,NaN,NaN,NaN,NaN
4,966504539322,7063DJR,MAZDA,MAZDA 6,Saudi,AR,PAC Brakes,Loyal,0,NaN,PAC,NaN,NaN,NaN,NaN


In [91]:
resty_df.shape

(430379, 15)

In [80]:
mdl_churn_predicted_df_extended.head()

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN
0,00000540-7986-4B85-A946-E1A9D913344F__F40BF538...,0,0.699744,00000540-7986-4B85-A946-E1A9D913344F,F40BF538-A9A6-49E3-9623-9301AE9FD953,TARIQ ABDUL AZIZ,0534747726,1065KHD,TMAJ28130MJ152970
1,00000847-CECD-43FB-A2F9-4D4B44A8BFFC__FF645321...,0,0.708705,00000847-CECD-43FB-A2F9-4D4B44A8BFFC,FF645321-C6D1-4864-91B7-F00EA37C30DB,MR. ALI MOHAMMAD ASIRI,0507340298,9664EHJ,6T1BF9FK7GX648439
2,00000E2B-FD56-4F34-A54C-159E564F5F79__FF8A2477...,0,0.679837,00000E2B-FD56-4F34-A54C-159E564F5F79,FF8A2477-4270-4547-B4CB-5706EA4479A0,MOHAMAD,0548121618,2250ASR,KNAF241BEKA835999
3,00001CA5-294A-4B26-A4A1-A77D7B5B18CC__9946561B...,1,0.637760,00001CA5-294A-4B26-A4A1-A77D7B5B18CC,9946561B-097B-441A-BB89-95AF12D27759,MR.FAIF,0571317927,8123UEB,KMHDT41B5BU252737
4,00001DF1-C75B-4CFA-9D95-FB370E5B95A7__E83698A7...,0,0.593648,00001DF1-C75B-4CFA-9D95-FB370E5B95A7,E83698A7-E358-42E9-9052-57664B17C455,HAMAM ALI,0557546128,3902NSB,VR3EF9HPANJ529993


In [109]:
filter_mobile = mdl_churn_predicted_df_extended["PlateNumber"].isin(resty_df["PlateNumber"].unique())
mdl_churn_predicted_df_extended[filter_mobile].head()

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN
9,000038EA-4314-421F-96D1-6894A5736CF6__3A210394...,0,0.454255,000038EA-4314-421F-96D1-6894A5736CF6,3A210394-0384-4DA7-8B0F-C01D83044449,MR.SULTAN,0505555671,9867JGD,JTNB14HK8K3096041
10,00003B03-AFE4-4DBE-B746-8D706FD771B8__8290FCEE...,0,0.782657,00003B03-AFE4-4DBE-B746-8D706FD771B8,8290FCEE-A8E3-4D69-A892-21A160C45DD6,FAISAL,0504689489,3562KAB,MR0EX12G2F2358216
23,000083D3-935E-4E89-AA93-2526C727FF76__EDD6D7BA...,1,0.124232,000083D3-935E-4E89-AA93-2526C727FF76,EDD6D7BA-1B05-4997-9AC9-53F4820265DB,BASSAM,0569195819,3428KAR,MHFBA8FS3N1061197
32,0000AD06-2D25-4EC8-BCCE-75B65ADB6110__2F8AABB7...,0,0.666305,0000AD06-2D25-4EC8-BCCE-75B65ADB6110,2F8AABB7-5CDB-4C4D-9739-FD387AD42F2B,MR. SAAD AL GATANI,0508887445,1753NBD,1GNSK7EC8HR514014
34,0000C428-6317-4E91-80A5-EB65BD4A99C0__FF66F89D...,0,0.635672,0000C428-6317-4E91-80A5-EB65BD4A99C0,FF66F89D-49CB-49B4-9993-A526BBE5C9E8,MUHAMMAD IMRAN,0574042256,3142LKJ,KMHDG41C0GU428081


In [93]:
mdl_churn_predicted_df_extended[filter_mobile].shape

(454979, 9)

In [112]:
output_df = mdl_churn_predicted_df_extended[filter_mobile].sort_values("churn_probability", ascending=False).drop_duplicates(subset=["PlateNumber", "Mobile"])
output_df.shape

(447224, 9)

In [128]:
resty_PAC_df["Mobile"] = resty_PAC_df["Mobile"].astype("string")
resty_PE_df["Mobile"] = resty_PE_df["Mobile"].astype("string")

In [123]:
mdl_churn_predicted_df_extended["Mobile"] = mdl_churn_predicted_df_extended["Mobile"].str.zfill(10)
mdl_churn_predicted_df_extended["Mobile"] = mdl_churn_predicted_df_extended["Mobile"].str.slice_replace(stop=1, repl='966')

In [129]:
output_PAC_df = pd.merge(
    resty_PAC_df,
    mdl_churn_predicted_df_extended[["Mobile", "PlateNumber", "CustomerName", "churn_probability"]],
    on=["PlateNumber", "Mobile"],
    how="left"
)
output_PE_df = pd.merge(
    resty_PE_df,
    mdl_churn_predicted_df_extended[["Mobile", "PlateNumber", "CustomerName", "churn_probability"]],
    on=["PlateNumber", "Mobile"],
    how="left"
)
output_PE_df.head()

,Mobile,PlateNumber,Make,Model,Nationality,PreferredLanguage,Segment,IsChinesePMS,DueOil,DuePMS,DueAirFilter,IsPromoHunter,LastPromoTaken,file,CustomerName,churn_probability
0,966500004857,9664HLB,TOYOTA,FJ CRUISER,Others,EN,Uncommitted,0,1,0,0,0,NaN,PE,ABDULAH,0.803952
1,966500004935,6008VVD,TOYOTA,LAND CRUISER,Saudi,AR,Uncommitted,0,1,0,0,0,NaN,PE,MR. MAJED ALI,0.778184
2,966500006050,0515ZHN,MERCEDES BENZ,S500,Others,AR,Uncommitted,0,1,0,0,0,NaN,PE,PRINCE ALANOD,0.758651
3,966500007572,6969ESD,LEXUS,ES350,Others,AR,Uncommitted,0,1,0,0,0,NaN,PE,MR.APALLAH,0.778399
4,966500008619,1632HGA,MERCURY,GRAND MARQUIS,Saudi,AR,Uncommitted,0,1,0,0,0,NaN,PE,BANDIR,0.807650


In [130]:
output_PE_df.isna().sum()

Mobile                    0
PlateNumber               0
Make                     68
Model                    39
Nationality               0
PreferredLanguage         0
Segment                   0
IsChinesePMS              0
DueOil                    0
DuePMS                    0
DueAirFilter              0
IsPromoHunter             0
LastPromoTaken       394138
file                      0
CustomerName            650
churn_probability       650
dtype: int64

In [131]:
output_PE_df.shape

(424801, 16)

In [133]:
output_PE_df.to_excel("../../../churn_PE_20250401.xlsx", engine="openpyxl", sheet_name="PE")
output_PAC_df.to_excel("../../../churn_PAC_20250401.xlsx", engine="openpyxl", sheet_name="PAC")

In [114]:
output_df[["Mobile", "PlateNumber", "CustomerName", "churn_probability"]].head()

,Mobile,PlateNumber,CustomerName,churn_probability
2790624,0567404142,1321GNR,MONSOUR AL OTAIBI,0.957142
2316948,0555768867,4576ZUD,SUMAYA SAAD ASHAMARI,0.955840
1785753,0505749274,1506RZD,KHALID,0.955580
1820377,0544440633,9095HDD,HASSAN,0.955373
1776946,0591516999,3865RTD,MR ALI,0.955372


In [107]:
x = mdl_churn_predicted_df_extended[filter_mobile][["PlateNumber", "Mobile"]].value_counts()
x[x > 1].sum()

15304

In [106]:
filter_plate = mdl_churn_predicted_df_extended["PlateNumber"] == "3183RSD"
mdl_churn_predicted_df_extended[filter_plate]

,_id,is_active,churn_probability,customer_id,customer_vehicle_id,CustomerName,Mobile,PlateNumber,VIN
187659,0E2143C2-ABBE-4D9E-8D99-D27FCC317D09__9756A6CE...,0,0.708705,0E2143C2-ABBE-4D9E-8D99-D27FCC317D09,9756A6CE-E6B8-47B4-9F70-EFD53EF2CA5A,MR HUSSEIN,0596610266,3183RSD,MHD741F0JU604001
436700,20CCD0ED-805E-440B-B2FD-877E4BF6F073__8573778B...,1,0.211448,20CCD0ED-805E-440B-B2FD-877E4BF6F073,8573778B-916E-4961-A442-0F2CC2E215DE,MR.HUSEEIN MOSA,0596610266,3183RSD,KMHD741F0JU604001
467152,2318057C-543E-4B79-8885-E9C4EAFCC830__57A118DF...,0,0.708705,2318057C-543E-4B79-8885-E9C4EAFCC830,57A118DF-434A-48EC-BC8B-A1A020A4F31F,MR. HUSSEIN,0596610266,3183RSD,KMHD741F0JU604001
1621795,7A0442B7-42EE-467D-9F74-8552E67CA3C2__63F3B85A...,0,0.708705,7A0442B7-42EE-467D-9F74-8552E67CA3C2,63F3B85A-03F2-4281-812A-10444F67524D,"MR,HUSSEIN KUBRANI",0596610266,3183RSD,KMHD741FOJU604001


In [88]:
resty_df.shape

(12771, 10)